## Visualize Quality Metrics

Basically we just need to write some scaffolding to download all the metrics from CSD3 and then turn this into a pretty dictionary / table (you do you), to easily visualize. Easy peasy!

In [8]:
import os
import json
from pathlib import Path
from typing import Any

from huggingface_hub import snapshot_download
import matplotlib.pyplot as plt
import pandas as pd

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

FONT_SIZES = {"small": 14, "medium": 18, "large": 24}

PLOT_PARAMS = {
    "font.family": "serif",
    "font.serif": ["Times"],
    "font.size": FONT_SIZES.get("medium"),
    "axes.titlesize": FONT_SIZES.get("large"),
    "axes.labelsize": FONT_SIZES.get("large"),
    "xtick.labelsize": FONT_SIZES.get("large"),
    "ytick.labelsize": FONT_SIZES.get("large"),
    "legend.fontsize": FONT_SIZES.get("medium"),
    "figure.titlesize": FONT_SIZES.get("medium"),
    "text.usetex": True,
}

COLORS = {
    "blue": "#D1F9F1",
    "crest": "#FFE2C8",
    "cherry": "#F2CAD8",
    "purple": "#F2ECF8",
    "green": "#DFF2EA",
    "indigo": "#EBEDFB",
    "heritage": "#85B09A",
    "warm_blue": "#00BDB6",
    "warm_crest": "#FFC392",
    "slate": "#B5BDC8",
}

plt.rcParams.update(PLOT_PARAMS)

In [9]:
def download_hf_files(repo_id: str, *, directory_path: str, local_path: Path):
    """Download files from a Hugging Face repository to a local directory."""
    if not local_path.exists():
        local_path.mkdir(parents=True, exist_ok=True)
    data_path = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=local_path,
        allow_patterns=[directory_path + "/*"],
    )
    return Path(data_path) / directory_path

In [10]:
metric_files = download_hf_files(
    "ljvmiranda921/mtep-intrinsic-metrics",
    directory_path="csd3",
    local_path=Path("./data/"),
)

Returning existing local_dir `data` as remote repo cannot be accessed in `snapshot_download` ((ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 92e60342-0753-409e-a2aa-519a42066001)')).


In [11]:
def parse_metrics(metrics_dir: Path) -> list[dict]:
    """Parse metric files and extract language, model, and data.

    Expected filename format: msde-S1-{language}_{model}_intrinsic_metrics
    Example: msde-S1-ar_mistralai__Mistral-Small-24B-Instruct-2501_intrinsic_metrics
    """
    metrics_data = []
    for file in metrics_dir.glob("*.json"):
        filename = file.stem
        parts = filename.replace("msde-S1-", "").replace("_intrinsic_metrics", "")
        language, model_raw = parts.split("_", 1)
        model = model_raw.replace("__", "/")

        with open(file, "r") as f:
            data = json.load(f)

        metrics_data.append(
            # fmt: off
            {
                "language": language,
                "model": model,
                "prompts_distinct_ri": data.get("distinct_ri", {}).get("prompts_distinct_ri"),
                "responses_distinct_ri": data.get("distinct_ri", {}).get("prompts_distinct_ri"),
                "rubric_score": data.get("reward_model", {}).get("average_rubric_score"),
                "perplexity": data.get("perplexity", {}).get("average_perplexity"),
            }
            # fmt: on
        )

    return metrics_data

In [12]:
def get_per_instance_metric(
    language: str, model: str, metric: str, metrics_dir: Path = Path("./data/csd3/")
) -> list[Any]:
    """Get per-instance results for a specific language, model, and metric."""
    model_filename = model.replace("/", "__")

    filename = f"msde-S1-{language}_{model_filename}_intrinsic_metrics.json"
    filepath = metrics_dir / filename

    if not filepath.exists():
        print(f"Warning: File not found: {filepath}")
        return []

    with open(filepath, "r") as f:
        data = json.load(f)

    if metric == "perplexity":
        perplexity_data = data.get("perplexity", {})
        return perplexity_data.get("per_instance_perplexity", [])
    elif metric == "rubric_score":
        reward_model_data = data.get("reward_model", {})
        return reward_model_data.get("per_instance_rubric_scores", [])
    else:
        print(
            f"Warning: Unsupported metric '{metric}'. Supported: 'perplexity', 'rubric_score'"
        )
        return []

In [13]:
df = (
    pd.DataFrame(parse_metrics(metric_files))
    .sort_values(by=["language", "model"])
    .reset_index(drop=True)
)
# Just use ar, cs, de for now
df = df[df["language"].isin(["ar", "cs", "de"])].reset_index(drop=True)
# Remove Mistral models
df = df[~df["model"].str.contains("Mistral")].reset_index(drop=True)

df

,language,model,prompts_distinct_ri,responses_distinct_ri,rubric_score,perplexity
0,ar,CohereLabs/aya-expanse-32b,0.692789,0.692789,3.9640,4.337098
1,ar,cohere-command-a,0.690184,0.690184,3.9956,5.405694
2,ar,google/gemma-3-12b-it,0.721363,0.721363,3.7736,4.431461
3,ar,google/gemma-3-27b-it,0.717244,0.717244,3.9315,4.403990
4,ar,google/gemma-3-4b-it,0.727739,0.727739,3.4699,5.518703
5,ar,gpt-4o-mini-2024-07-18,0.703685,0.703685,3.5159,NaN
6,ar,ibm-granite/granite-4.0-1b,0.703923,0.703923,2.4633,18604.332290
7,ar,ibm-granite/granite-4.0-micro,0.741429,0.741429,3.0330,12.454267
8,ar,meta-llama/Llama-3.1-70B-Instruct,0.700566,0.700566,2.7185,7.004205
9,ar,meta-llama/Llama-3.1-8B-Instruct,0.707724,0.707724,1.7314,62410.433995


In [14]:
import numpy as np

# Recompute perplexity from per-instance data, excluding NaN values
print("Recomputing perplexity values...")
for idx, row in df.iterrows():
    per_instance = get_per_instance_metric(row["language"], row["model"], "perplexity")

    # Extract perplexity values and filter out NaNs
    values = []
    for item in per_instance:
        if isinstance(item, dict):
            val = item.get("perplexity")
        else:
            val = item

        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            values.append(val)

    # Compute mean or set to NaN if no valid values
    if values:
        df.at[idx, "perplexity"] = np.mean(values)
    else:
        df.at[idx, "perplexity"] = np.nan

print("✓ Done!")
df

Recomputing perplexity values...
✓ Done!


,language,model,prompts_distinct_ri,responses_distinct_ri,rubric_score,perplexity
0,ar,CohereLabs/aya-expanse-32b,0.692789,0.692789,3.9640,4.337098
1,ar,cohere-command-a,0.690184,0.690184,3.9956,5.405694
2,ar,google/gemma-3-12b-it,0.721363,0.721363,3.7736,4.431461
3,ar,google/gemma-3-27b-it,0.717244,0.717244,3.9315,4.403990
4,ar,google/gemma-3-4b-it,0.727739,0.727739,3.4699,5.518703
5,ar,gpt-4o-mini-2024-07-18,0.703685,0.703685,3.5159,8.395005
6,ar,ibm-granite/granite-4.0-1b,0.703923,0.703923,2.4633,18604.332290
7,ar,ibm-granite/granite-4.0-micro,0.741429,0.741429,3.0330,12.454267
8,ar,meta-llama/Llama-3.1-70B-Instruct,0.700566,0.700566,2.7185,7.004205
9,ar,meta-llama/Llama-3.1-8B-Instruct,0.707724,0.707724,1.7314,62410.433995
